# Marathon dataset, first look

Sections match the figure captions in `Marathon_data_review.docx`.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# 80k rows to work with, 20k held back by kaggle
tr = pd.read_csv('../data/train.csv')
te = pd.read_csv('../data/test.csv')

tr.shape, te.shape

In [ ]:
# no medal_outcome in test -> can't score anything with it
set(tr.columns) - set(te.columns)

In [ ]:
# three groups of columns. everything below leans on this split.
BEHAVIOUR = ['motivation_level','mental_preparation_score','training_adherence_pct',
             'consecutive_weeks_no_miss','training_streak_days','missed_workout_pct',
             'goal_completion_rate','run_club_attendance_rate','warmup_adherence_pct',
             'stretching_adherence_pct','early_morning_run_frequency',
             'weather_condition_training_pct','nutrition_score','hydration_consistency',
             'sleep_hours_avg','recovery_score']

BODY = ['age','running_experience_months','previous_marathon_count','weekly_mileage_miles',
        'runs_per_week','long_run_distance_km','speed_work_sessions_per_week',
        'rest_days_per_week','cross_training_hours_per_week','resting_heart_rate_bpm',
        'vo2_max','bmi','injury_count']

SPOILERS = ['target_finish_time_minutes','personal_best_minutes']

## 1. Target time

In [ ]:
# how far off people finish from the time they aimed for
d = tr.dropna(subset=['actual_finish_time_minutes']).copy()
d['gap'] = d.actual_finish_time_minutes - d.target_finish_time_minutes

d.gap.describe().round(1)

In [ ]:
# min gap is +11, so nobody beats their own target. that can't be a feature.
(d.gap <= 0).sum(), round(d.actual_finish_time_minutes.corr(d.target_finish_time_minutes), 3)

## 2. Mileage columns

In [ ]:
# is any column mostly the same number repeated?
for c in ['weekly_mileage_km', 'long_run_distance_km']:
    top = tr[c].mode()[0]
    print(c, '=', top, 'in', f'{(tr[c] == top).mean():.0%}', 'of rows')

In [ ]:
# should be 1.609 everywhere
(tr.weekly_mileage_km / tr.weekly_mileage_miles).describe().round(2)

## 3. Nulls

In [ ]:
# both are null for a reason, not by accident
print(tr[tr.personal_best_minutes.isna()].previous_marathon_count.unique())
print(tr[tr.injury_severity.isna()].injury_count.unique())

## 4. Drop-outs (Figure 1)

In [ ]:
# no finish time = didn't finish. see if it tracks injuries.
tr['dnf'] = tr.actual_finish_time_minutes.isna().astype(int)
print(f'{tr.dnf.mean():.1%} have no finish time')

by_injury = tr[tr.injury_count <= 4].groupby('injury_count').dnf.mean()
by_injury.round(3)

In [ ]:
# and by programme: beginners quit twice as often
tr.groupby('training_program').dnf.mean().round(3)

In [ ]:
# figure 1
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(by_injury.index, by_injury * 100, marker='o', color='#2a78d6')
ax.set_xlabel('injuries during preparation')
ax.set_ylabel('% with no finish time')
ax.set_title('Drop-out rate rises with injuries')
ax.spines[['top', 'right']].set_visible(False)

## 5. What explains the time, and what explains the gap (Figure 2)

In [ ]:
# one model per group of columns, same folds for all of them so they compare
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(4, shuffle=True, random_state=0)

def r2(cols, target):
    m = HistGradientBoostingRegressor(random_state=0)
    return cross_val_score(m, d[cols], target, cv=cv, scoring='r2').mean()

In [ ]:
# run it for each combination, predicting the absolute time
blocks = {
    'with the spoilers': BEHAVIOUR + BODY + SPOILERS,
    'body + behaviour': BEHAVIOUR + BODY,
    'body only': BODY,
    'behaviour only': BEHAVIOUR,
    'experience alone': ['running_experience_months'],
}

on_time = pd.Series({name: round(r2(cols, d.actual_finish_time_minutes), 3)
                     for name, cols in blocks.items()})
on_time

In [ ]:
# now the same behaviour columns against the gap instead. big difference.
on_gap = r2(BEHAVIOUR, d.gap)
round(on_time['behaviour only'], 3), round(on_gap, 3)

In [ ]:
# figure 2
fig, ax = plt.subplots(figsize=(6, 2.4))
ax.barh(['gap to own target', 'absolute finish time'],
        [on_gap, on_time['behaviour only']], color=['#eb6834', '#2a78d6'])
ax.set_xlabel('cross-validated R2, behaviour block only')
ax.set_title('The behaviour block explains the gap, not the clock')
ax.spines[['top', 'right']].set_visible(False)

The target already contains how good the runner is, so subtracting it removes the ability
term and leaves how the preparation and the day went.

## 6. Race day and the medal (Figures 3 and 4)

In [ ]:
# things the runner doesn't control also move the gap
by_weather = d.groupby('marathon_weather').gap.mean().sort_values()
by_course = d.groupby('course_difficulty').gap.mean().sort_values()

pd.concat([by_weather, by_course]).round(1)

In [ ]:
# figure 3
fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))

axes[0].bar(by_weather.index, by_weather.values, color='#2a78d6')
axes[0].set_title('by race-day weather')
axes[0].set_ylabel('minutes behind target')

axes[1].bar(by_course.index, by_course.values, color='#1baf7a')
axes[1].set_title('by course')

for ax in axes:
    ax.set_ylim(30, 46)
    ax.tick_params(axis='x', rotation=25)
    ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()

In [ ]:
# same blocks again, but predicting the medal
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold

d['medal'] = (d.medal_outcome > 0).astype(int)
scv = StratifiedKFold(4, shuffle=True, random_state=0)

def auc(cols):
    m = HistGradientBoostingClassifier(random_state=0)
    return cross_val_score(m, d[cols], d.medal, cv=scv, scoring='roc_auc').mean()

pd.Series({'body + behaviour': auc(BEHAVIOUR + BODY),
           'body only': auc(BODY),
           'behaviour only': auc(BEHAVIOUR)}).round(3)

In [ ]:
# maybe fast runners are just also disciplined? split by speed first.
d['speed'] = pd.qcut(d.actual_finish_time_minutes, 5,
                     labels=['fastest', 'fast', 'middle', 'slow', 'slowest'])
d['adh3'] = pd.qcut(d.training_adherence_pct, 3, labels=['low', 'medium', 'high'])

within_speed = pd.crosstab(d.speed, d.adh3, values=d.medal, aggfunc='mean')
within_speed.round(3)

In [ ]:
# figure 4
ax = (within_speed * 100).plot.bar(figsize=(7, 3), rot=0,
                                   color=['#2a78d6', '#1baf7a', '#eb6834'])
ax.set_ylabel('% winning a medal')
ax.set_title('Consistency still pays inside every speed band')
ax.legend(title='adherence', frameon=False)
ax.spines[['top', 'right']].set_visible(False)

## 7. What the behaviour columns measure

In [ ]:
# which behaviour columns are just measuring the same thing twice
c = tr[BEHAVIOUR].corr().abs()
pairs = c.where(np.triu(np.ones(c.shape), 1).astype(bool)).stack()

pairs.sort_values(ascending=False).head(6).round(2)

Adherence and missed workouts are the same thing twice (0.89), so drop one before clustering.
Motivation shows up in behaviour (early runs, run club) but never in results.